# 🎫 LangGraph Ticket Escalation Router

## What We're Building

A support-ticket routing workflow that:
1. **Classifies incoming tickets** — a local keyword classifier determines severity and category
2. **Routes** — simple tickets go to auto-resolve; complex/urgent ones escalate
3. **Auto-resolves** — LLM generates a helpful response for routine issues
4. **Escalates** — formats a structured escalation brief for a human agent

## Architecture

```
Incoming Ticket
       │
       ▼
┌──────────────┐
│  Classify     │
│  (local)      │
└──────┬───────┘
       │
  ┌────┴────┐
  │ Route?  │
  └────┬────┘
       │
  ┌────┴─────────────┐
  │                  │
  ▼                  ▼
┌──────────┐   ┌──────────┐
│ Auto-    │   │ Escalate │
│ Resolve  │   │ to Human │
│ (LLM)    │   │ (LLM)    │
└──────────┘   └──────────┘
```

## Stack
- **LangGraph** — conditional routing based on classification
- **Local classifier** — keyword/rule-based severity classification (no ML model needed)
- **Ollama** — LLM for auto-response generation and escalation briefs

## Step 1 — Imports & Configuration

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:4b", temperature=0.2)
print("All imports successful!")

## Step 2 — Sample Support Tickets

We test with a mix of simple (auto-resolvable) and complex (escalation-worthy) tickets.

In [ ]:
SAMPLE_TICKETS = [
    {
        "ticket_id": "TK-1001",
        "subject": "Password reset not working",
        "body": "I tried to reset my password using the forgot password link but the email never arrives. I've checked my spam folder. My email is john@example.com.",
        "customer_tier": "standard"
    },
    {
        "ticket_id": "TK-1002",
        "subject": "URGENT: Production system down",
        "body": "Our entire production environment has been down for 2 hours. All API endpoints returning 503. This is affecting 50,000 users. We need immediate help. Our SLA guarantees 99.9% uptime.",
        "customer_tier": "enterprise"
    },
    {
        "ticket_id": "TK-1003",
        "subject": "How to export data to CSV",
        "body": "Hi, I'd like to export my dashboard data to CSV format. I can't find the option. Can you help?",
        "customer_tier": "standard"
    },
    {
        "ticket_id": "TK-1004",
        "subject": "Data breach suspected",
        "body": "We noticed unauthorized access to several user accounts yesterday. IP addresses from unknown locations accessed admin panels. We suspect a security breach and need your security team involved immediately.",
        "customer_tier": "enterprise"
    },
]

print(f"Loaded {len(SAMPLE_TICKETS)} sample tickets")
for t in SAMPLE_TICKETS:
    print(f"  • {t['ticket_id']}: {t['subject']}")

## Step 3 — Define State & Local Classifier

The local classifier uses keyword matching and rules to determine:
- **Category**: billing, technical, security, general
- **Severity**: low, medium, high, critical
- **Route decision**: auto_resolve or escalate

In [ ]:
class TicketState(TypedDict):
    ticket_id: str
    subject: str
    body: str
    customer_tier: str
    category: str
    severity: str
    route: str               # "auto_resolve" or "escalate"
    response: str            # Auto-generated response or escalation brief


# --- Local keyword classifier ---
ESCALATION_KEYWORDS = {
    "critical": ["down", "outage", "breach", "unauthorized", "security", "hack",
                 "data loss", "compliance", "legal"],
    "high": ["urgent", "production", "sla", "enterprise", "50,000", "immediately"],
}

CATEGORY_KEYWORDS = {
    "security": ["breach", "unauthorized", "hack", "security", "suspicious"],
    "billing": ["invoice", "payment", "charge", "refund", "billing"],
    "technical": ["error", "bug", "crash", "down", "not working", "api", "503"],
}


def classify_ticket_local(subject: str, body: str, tier: str) -> dict:
    """Rule-based ticket classifier."""
    text = f"{subject} {body}".lower()

    # Determine severity
    severity = "low"
    for kw in ESCALATION_KEYWORDS["critical"]:
        if kw in text:
            severity = "critical"
            break
    if severity == "low":
        for kw in ESCALATION_KEYWORDS["high"]:
            if kw in text:
                severity = "high"
                break
    if severity == "low" and ("password" in text or "reset" in text or "export" in text):
        severity = "low"
    elif severity == "low":
        severity = "medium"

    # Determine category
    category = "general"
    for cat, keywords in CATEGORY_KEYWORDS.items():
        for kw in keywords:
            if kw in text:
                category = cat
                break
        if category != "general":
            break

    # Route decision
    if severity in ("critical", "high") or tier == "enterprise" or category == "security":
        route = "escalate"
    else:
        route = "auto_resolve"

    return {"category": category, "severity": severity, "route": route}


print("Local classifier defined")
# Quick test
test = classify_ticket_local("URGENT: system down", "production is down", "enterprise")
print(f"Test classification: {test}")

## Step 4 — Define Graph Nodes

| Node | Role |
|------|------|
| `classify` | Local classifier determines category, severity, route |
| `auto_resolve` | LLM generates a helpful customer response |
| `escalate` | LLM formats a structured escalation brief for human agents |

In [ ]:
def classify(state: TicketState) -> dict:
    """Classify the ticket using the local rule-based classifier."""
    print(f"🏷️ Node: classify — Ticket {state['ticket_id']}")
    result = classify_ticket_local(state["subject"], state["body"], state["customer_tier"])
    print(f"     Category: {result['category']}")
    print(f"     Severity: {result['severity']}")
    print(f"     Route:    {result['route']}")
    return result


auto_prompt = ChatPromptTemplate.from_template(
    """You are a friendly customer support agent. Write a helpful response
to this support ticket.

Ticket ID: {ticket_id}
Subject: {subject}
Customer Message: {body}

Category: {category}
Severity: {severity}

Write a professional, empathetic response that:
- Acknowledges the issue
- Provides clear step-by-step instructions to resolve it
- Offers to help further if needed

Keep it concise and friendly."""
)


def auto_resolve(state: TicketState) -> dict:
    """Generate an automated response for simple tickets."""
    print(f"🤖 Node: auto_resolve — Generating response for {state['ticket_id']}")
    chain = auto_prompt | llm | StrOutputParser()
    response = chain.invoke({
        "ticket_id": state["ticket_id"],
        "subject": state["subject"],
        "body": state["body"],
        "category": state["category"],
        "severity": state["severity"],
    })
    return {"response": f"[AUTO-RESOLVED]\n\n{response}"}


escalate_prompt = ChatPromptTemplate.from_template(
    """You are a support triage specialist. Create an escalation brief
for a human agent to handle this ticket.

Ticket ID: {ticket_id}
Subject: {subject}
Customer Message: {body}
Customer Tier: {customer_tier}
Category: {category}
Severity: {severity}

Create an escalation brief with:
- SUMMARY (1-2 sentences)
- IMPACT ASSESSMENT (who/what is affected)
- RECOMMENDED TEAM (e.g., Security, DevOps, Account Management)
- SUGGESTED PRIORITY (P1/P2/P3)
- INITIAL INVESTIGATION STEPS
- CUSTOMER COMMUNICATION (what to tell the customer while investigating)

Be concise and actionable."""
)


def escalate(state: TicketState) -> dict:
    """Generate a structured escalation brief."""
    print(f"🚨 Node: escalate — Preparing brief for {state['ticket_id']}")
    chain = escalate_prompt | llm | StrOutputParser()
    brief = chain.invoke({
        "ticket_id": state["ticket_id"],
        "subject": state["subject"],
        "body": state["body"],
        "customer_tier": state["customer_tier"],
        "category": state["category"],
        "severity": state["severity"],
    })
    return {"response": f"[ESCALATED — {state['severity'].upper()}]\n\n{brief}"}


def route_ticket(state: TicketState) -> Literal["auto_resolve", "escalate"]:
    """Conditional edge: route based on classification."""
    return state["route"]


print("All nodes defined")

## Step 5 — Build Graph with Conditional Routing

In [ ]:
workflow = StateGraph(TicketState)

workflow.add_node("classify", classify)
workflow.add_node("auto_resolve", auto_resolve)
workflow.add_node("escalate", escalate)

workflow.set_entry_point("classify")
workflow.add_conditional_edges(
    "classify",
    route_ticket,
    {"auto_resolve": "auto_resolve", "escalate": "escalate"}
)
workflow.add_edge("auto_resolve", END)
workflow.add_edge("escalate", END)

app = workflow.compile()
print("Ticket escalation router compiled")

## Step 6 — Process All Sample Tickets

In [ ]:
results = []

for ticket in SAMPLE_TICKETS:
    print("\n" + "─"*60)
    result = app.invoke({
        "ticket_id": ticket["ticket_id"],
        "subject": ticket["subject"],
        "body": ticket["body"],
        "customer_tier": ticket["customer_tier"],
        "category": "",
        "severity": "",
        "route": "",
        "response": "",
    })
    results.append(result)

print("\n" + "="*60)
print("📊 ROUTING SUMMARY")
print("="*60)
for r in results:
    status = "🤖 AUTO" if "AUTO-RESOLVED" in r["response"] else "🚨 ESCALATED"
    print(f"  {r['ticket_id']}: {status} | {r['category']} | {r['severity']}")

## Step 7 — Inspect Individual Responses

In [ ]:
for r in results:
    print("\n" + "="*60)
    print(f"📋 {r['ticket_id']}: {r['route'].upper()}")
    print(f"   Category: {r['category']} | Severity: {r['severity']}")
    print("="*60)
    print(r["response"])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Local classifier** | Keyword/rule-based — fast, no API calls, deterministic |
| **Conditional routing** | `add_conditional_edges` sends tickets to different nodes |
| **Auto-resolution** | LLM generates helpful responses for routine issues |
| **Escalation briefs** | LLM formats structured handoff for human agents |
| **Hybrid approach** | Local rules for routing + LLM for response quality |

## Why Hybrid (Local + LLM) Routing?

- **Speed**: Local classifier routes instantly — no LLM latency for triage
- **Reliability**: Keyword rules are deterministic and auditable
- **Cost**: Only auto-resolve and escalation steps use the LLM
- **Safety**: Critical tickets always escalate — no LLM hallucination risk

## 🔧 Extensions

- **ML classifier**: Replace keywords with a trained text classifier
- **Priority queue**: Escalated tickets enter a priority queue with SLA timers
- **Feedback loop**: Track if auto-resolved tickets get reopened
- **Multi-language**: Add language detection and route to language-specific teams